# Object tracking

Follow annotated moving boxes using the original greedy IoU tracker.

Run with the repository virtual environment. These are synthetic research experiments; units and assumptions are explicit below.

## Sequence

This tracker uses no pretrained detector or appearance embeddings.

In [ ]:
import matplotlib.pyplot as plt

from autonomous_vehicles.evaluation.tracking import TrackingEvaluator
from autonomous_vehicles.perception.boxes import Box
from autonomous_vehicles.perception.sequences import MovingBox, moving_sequence
from autonomous_vehicles.perception.tracking import IoUTracker, TrackerConfig

objects = (
    MovingBox("car-a", Box(5, 10, 20, 22), vx=2),
    MovingBox("car-b", Box(90, 35, 105, 47), vx=-1),
)
sequence = moving_sequence(objects, frames=25, dropout=0.1, noise=0.3, seed=42)

## Inspect annotations

Targets preserve identity even when synthetic detections drop out.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, index in zip(axes, (0, 10, 20), strict=True):
    frame = sequence[index]
    ax.imshow(frame.image)
    ax.set_title(f"Frame {index}: {len(frame.detections)} detections")
    for target in frame.targets:
        ax.text(*target.box.center, target.identifier, color="white")
plt.tight_layout()
plt.show()

## Track and evaluate

The evaluator matches boxes by IoU and records identity changes against annotated IDs.

In [ ]:
tracker = IoUTracker(TrackerConfig(maximum_misses=2))
evaluator = TrackingEvaluator(iou_threshold=0.5)
records = []
for frame in sequence:
    tracks = tracker.update(frame.detections, dt=1)
    evaluator.update(tuple(t for t in tracks if t.misses == 0), frame.targets)
    records.append((frame.time, tracks))
print(evaluator.summary())
print("MOTA-like summary:", evaluator.summary().mota_like)

## Plot ID histories

Missing observations are distinct from deleted tracks. The lifecycle retains missed tracks briefly.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ids = sorted({track.identifier for _, tracks in records for track in tracks})
for identifier in ids:
    samples = [
        (time, track.box.center[0])
        for time, tracks in records
        for track in tracks
        if track.identifier == identifier
    ]
    ax.plot([s[0] for s in samples], [s[1] for s in samples], marker=".", label=f"ID {identifier}")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Box center x (pixels)")
ax.legend()
plt.show()

## Stress dropout

Compare identical motion with increasing missing detections; these are synthetic observations.

In [ ]:
for dropout in (0.0, 0.2, 0.5):
    tracker = IoUTracker()
    evaluator = TrackingEvaluator()
    trial = moving_sequence(objects, frames=25, dropout=dropout, seed=42)
    for frame in trial:
        tracks = tracker.update(frame.detections)
        evaluator.update(tuple(track for track in tracks if track.misses == 0), frame.targets)
    summary = evaluator.summary()
    print(
        {
            "dropout": dropout,
            "misses": summary.misses,
            "switches": summary.identity_switches,
            "mota_like": summary.mota_like,
        }
    )

## Research record

Record the seed, configuration, dependency lock and observed failures with each experiment. Results here demonstrate the implemented model under its stated assumptions. They do not establish performance outside this synthetic distribution.